# 03_06_AdamW_＋_Warmup_＋_Cosine_

コード：`ch03/blocks/03_06_AdamW_＋_Warmup_＋_Cosine_.ipynb`

本文のコードをコードセルで直接実行します。

上からセルを実行してください。ローカルではコード一式と同じPython環境をJupyterのカーネルに選びます。Colabでコード一式が必要な場合は、準備セルで配布ZIPをアップロードします。実行確認の範囲は `VALIDATION_COLAB_20261110.md` を参照してください。


In [ ]:
import subprocess
import sys

# 既に導入済みなら不要な更新をせず、満たしていない依存だけを導入する。
subprocess.run([sys.executable, "-m", "pip", "install", *['numpy>=1.26,<3', 'torch>=2.6,<3', 'torchvision>=0.21,<1', 'mcp==2.3.0', 'PyYAML==6.0.3', 'opentelemetry-sdk==1.45.2', 'opentelemetry-exporter-otlp-proto-http==1.45.2']], check=True)


In [ ]:
# 直す前の書き方：lr=1e-4・warmup の 1 step 目が 0・lr を step() の後に読む・bias にも decay・cosine が 0 まで
import math, torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(0)
x = torch.linspace(-1, 1, 100).unsqueeze(1)
y = 2 * x + 1 + 0.05 * torch.randn_like(x)
perm = torch.randperm(100)
x, y = x[perm], y[perm]
n_tr = 80
train = DataLoader(TensorDataset(x[:n_tr], y[:n_tr]), batch_size=10, shuffle=True)

model   = nn.Linear(1, 1)
loss_fn = nn.MSELoss()
opt = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)   # 全パラメータに decay

EPOCHS      = 30
total_steps = EPOCHS * len(train)
warmup      = int(total_steps * 0.1)

def factor(step):
    if step < warmup:
        return step / warmup                       # step=0 で 0
    p = (step - warmup) / (total_steps - warmup)
    return 0.5 * (1 + math.cos(math.pi * p))       # 0 まで落ちる

sch = torch.optim.lr_scheduler.LambdaLR(opt, factor)

with torch.no_grad():
    print(f"開始: w={model.weight.item():.3f}  b={model.bias.item():.3f}  "
          f"loss={loss_fn(model(x[:n_tr]), y[:n_tr]).item():.3f}")
step = 0
for epoch in range(EPOCHS):
    for bx, by in train:
        w_before = model.weight.item()
        opt.zero_grad()
        loss_fn(model(bx), by).backward()
        opt.step()
        sch.step()
        step += 1
        logged = opt.param_groups[0]["lr"]             # step() の後に読む
        if step <= 3:
            print(f"step {step}: w の変化={model.weight.item() - w_before:+.2e}  ログの lr={logged:.2e}")
with torch.no_grad():
    print(f"終了: w={model.weight.item():.3f}  b={model.bias.item():.3f}  "
          f"loss={loss_fn(model(x[:n_tr]), y[:n_tr]).item():.3f}  （{total_steps} step）")
